In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os, sys

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")


from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema, get_features_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
features_schema = get_features_schema() 
mart_schema = get_mart_schema() 

from superstore_logger import get_superstore_logger, log_event
from superstore_backfill_utils import is_dry_run
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_features_products = get_superstore_logger("superstore_features_customers_products")


# -----------------------------
# Dry run
# -----------------------------
# The serving layer rebuilds from Gold and has no run-mode logic of its own —
# a replay or backfill simply produces different Gold for it to read. But it
# does write, so it must not write during a dry run. is_dry_run reads only the
# dry_run parameter and never raises, so a serving task cannot fail on the
# full_refresh gate that is enforced upstream where it belongs.
if is_dry_run(dbutils):
    log_event(
        logger_features_products,
        "INFO",
        "Dry run: skipping features_product write",
        dry_run=True,
        layer="Feature Engineering"
    )
    dbutils.notebook.exit("DRY_RUN_COMPLETED layer=features_product (no data written)")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_master,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="Feature Engineering"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
# Check if the required tables exist before creating the feature table
required_tables = [
    f"{catalog}.{gold_schema}.facts_orders",
    f"{catalog}.{gold_schema}.facts_sales"
]

missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, gold_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_features_products , "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create feature table.", layer="Feature Engineering")
else:
    # -------------------------------
    # Create or update features_product Table (Environment-aware)
    # -------------------------------
    # Check if the product features table already exists
    if spark.catalog.tableExists(f"{catalog}.{features_schema}.features_product"):
        # Table exists, perform MERGE to update existing rows and insert new ones
        spark.sql(f"""
            MERGE INTO {catalog}.{features_schema}.features_product AS target
            USING (
                -- 1. Base sales data per product
                WITH sales_base AS (
                    SELECT
                        s.product_id,
                        o.order_date,
                        s.sales AS total_sales,
                        s.profit AS total_profit,
                        s.discount AS avg_discount
                    FROM {catalog}.{gold_schema}.facts_sales s
                    JOIN {catalog}.{gold_schema}.facts_orders o
                        ON s.order_id = o.order_id
                ),

                -- 2. Sales in last 7 & 30 days
                sales_window AS (
                    SELECT
                        product_id,
                        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 7) THEN total_sales ELSE 0 END) AS sales_last_7days,
                        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 30) THEN total_sales ELSE 0 END) AS sales_last_30days
                    FROM sales_base
                    GROUP BY product_id
                ),

                -- 3. Sales Trend
                sales_trend AS (
                    SELECT
                        product_id,
                        CASE WHEN sales_last_30days = 0 THEN 0
                            ELSE ROUND((sales_last_7days / 7.0) / (sales_last_30days / 30.0), 2)
                        END AS sales_trend
                    FROM sales_window
                ),

                -- 4. Discount Sensitivity
                discount_sensitivity AS (
                    SELECT
                        product_id,
                        ROUND(AVG(avg_discount) * 100, 2) AS discount_sensitivity
                    FROM sales_base
                    GROUP BY product_id
                ),

                -- 5. Profit Volatility
                profit_volatility AS (
                    SELECT
                        product_id,
                        ROUND(VAR_SAMP(total_profit), 2) AS profit_volatility
                    FROM sales_base
                    GROUP BY product_id
                ),

                -- 6. Seasonality Index
                seasonality AS (
                    SELECT
                        product_id,
                        ROUND(AVG(monthly_sales) / NULLIF(AVG(overall_avg_sales), 0), 2) AS seasonality_index
                    FROM (
                        SELECT
                            product_id,
                            MONTH(order_date) AS month,
                            AVG(total_sales) AS monthly_sales,
                            AVG(AVG(total_sales)) OVER (PARTITION BY product_id) AS overall_avg_sales
                        FROM sales_base
                        GROUP BY product_id, MONTH(order_date)
                    ) t
                    GROUP BY product_id
                )

                SELECT
                    s.product_id,
                    CURRENT_DATE() AS feature_date,
                    s.sales_last_7days,
                    s.sales_last_30days,
                    st.sales_trend,
                    ds.discount_sensitivity,
                    pv.profit_volatility,
                    se.seasonality_index
                FROM sales_window s
                JOIN sales_trend st USING (product_id)
                JOIN discount_sensitivity ds USING (product_id)
                JOIN profit_volatility pv USING (product_id)
                JOIN seasonality se USING (product_id)
            ) AS source
            ON target.product_id = source.product_id
            AND target.feature_date = source.feature_date

            -- Update existing records if matched
            WHEN MATCHED THEN
                UPDATE SET
                    target.sales_last_7days = source.sales_last_7days,
                    target.sales_last_30days = source.sales_last_30days,
                    target.sales_trend = source.sales_trend,
                    target.discount_sensitivity = source.discount_sensitivity,
                    target.profit_volatility = source.profit_volatility,
                    target.seasonality_index = source.seasonality_index

            -- Insert new records if not matched
            WHEN NOT MATCHED THEN
                INSERT (product_id, feature_date, sales_last_7days, sales_last_30days, sales_trend,
                        discount_sensitivity, profit_volatility, seasonality_index)
                VALUES (source.product_id, source.feature_date, source.sales_last_7days, source.sales_last_30days,
                        source.sales_trend, source.discount_sensitivity, source.profit_volatility, source.seasonality_index)
        """)
    else:
        # Table does not exist, create it from scratch
        spark.sql(f"""
            -- Create the Product ML Features Table
            CREATE TABLE {catalog}.{features_schema}.features_product AS
            WITH sales_base AS (
                SELECT
                    s.product_id,
                    o.order_date,
                    s.sales AS total_sales,
                    s.profit AS total_profit,
                    s.discount AS avg_discount
                FROM {catalog}.{gold_schema}.facts_sales s
                JOIN {catalog}.{gold_schema}.facts_orders o
                    ON s.order_id = o.order_id
            ),
            sales_window AS (
                SELECT
                    product_id,
                    SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 7) THEN total_sales ELSE 0 END) AS sales_last_7days,
                    SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 30) THEN total_sales ELSE 0 END) AS sales_last_30days
                FROM sales_base
                GROUP BY product_id
            ),
            sales_trend AS (
                SELECT
                    product_id,
                    CASE WHEN sales_last_30days = 0 THEN 0
                        ELSE ROUND((sales_last_7days / 7.0) / (sales_last_30days / 30.0), 2)
                    END AS sales_trend
                FROM sales_window
            ),
            discount_sensitivity AS (
                SELECT
                    product_id,
                    ROUND(AVG(avg_discount) * 100, 2) AS discount_sensitivity
                FROM sales_base
                GROUP BY product_id
            ),
            profit_volatility AS (
                SELECT
                    product_id,
                    ROUND(VAR_SAMP(total_profit), 2) AS profit_volatility
                FROM sales_base
                GROUP BY product_id
            ),
            seasonality AS (
                SELECT
                    product_id,
                    ROUND(AVG(monthly_sales) / NULLIF(AVG(overall_avg_sales), 0), 2) AS seasonality_index
                FROM (
                    SELECT
                        product_id,
                        MONTH(order_date) AS month,
                        AVG(total_sales) AS monthly_sales,
                        AVG(AVG(total_sales)) OVER (PARTITION BY product_id) AS overall_avg_sales
                    FROM sales_base
                    GROUP BY product_id, MONTH(order_date)
                ) t
                GROUP BY product_id
            )
            SELECT
                s.product_id,
                CURRENT_DATE() AS feature_date,
                s.sales_last_7days,
                s.sales_last_30days,
                st.sales_trend,
                ds.discount_sensitivity,
                pv.profit_volatility,
                se.seasonality_index
            FROM sales_window s
            JOIN sales_trend st USING (product_id)
            JOIN discount_sensitivity ds USING (product_id)
            JOIN profit_volatility pv USING (product_id)
            JOIN seasonality se USING (product_id)
        """)
log_event(logger_features_products, "INFO", "Feature product table created successfully.", layer="Feature Engineering")
